In [ ]:
!pip install transformers
!pip install datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset,DatasetDict
from transformers import AutoTokenizer,TFAutoModelForSequenceClassification
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Experiments for research paper/Cyberbullying detection/train_merged.csv"
tweets_df = pd.read_csv(DATA_PATH)
tweets_df.head(10)

In [ ]:
!pip install contractions

In [ ]:
import re

# Remove escape characters
tweets_df['Comment_cleaned'] = tweets_df['Comment'].apply(lambda x: re.sub(r'\\[a-zA-Z]', ' ', x))

#Remove numbers
tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(lambda x: re.sub(r'\d+', ' ', x))

# Remove URLs and mentions (@username) from the tweets
tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(lambda x: re.sub(r'http\S+', ' ', x)) # Remove URLs
tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(lambda x: re.sub(r'@\w+', ' ', x)) # Remove mentions


import contractions

def expand_contractions(text):
    expanded_text = contractions.fix(text)
    return expanded_text

tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(expand_contractions)


# Remove punctuation and convert all characters to lowercase
tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(lambda x: re.sub(r'[^\w\s]', ' ', x)) # Remove punctuation
tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(lambda x: x.lower()) # Convert to lowercase



# Remove stop words
# import nltk
# nltk.download('stopwords')
# from nltk.corpus import stopwords
# stop_words = stopwords.words('english')
# tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

#Remove single alphabet
tweets_df['Comment_cleaned'] = tweets_df['Comment_cleaned'].apply(lambda x: re.sub(r'\b\w\b', '', x))

tweets_df.head(10)

In [ ]:
tweets_df.head()

In [ ]:
tweets_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Experiments for research paper/Cyberbullying detection/train_merged_cleaned.csv")

In [ ]:
from datasets import Dataset



ds = Dataset.from_pandas(tweets_df)
ds



In [ ]:
DATACLEAN_PATH = "/content/drive/MyDrive/Colab Notebooks/Experiments for research paper/Cyberbullying detection/train_merged_cleaned.csv"

In [ ]:
dataset = load_dataset('csv', data_files=DATACLEAN_PATH, split='train')

dataset

In [ ]:
train_test_valid = ds.train_test_split(test_size=.40)

test_valid = train_test_valid['test'].train_test_split(test_size=0.5)

train_test_valid_dataset = DatasetDict({
    'train': train_test_valid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']
    })


dataset = train_test_valid_dataset.remove_columns(['Date', 'Comment'])
dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("GroNLP/hateBERT")

In [ ]:
text = "Just checking tokenization"

output = tokenizer(text)

output


In [ ]:
tokens = tokenizer.convert_ids_to_tokens(output['input_ids'])
tokens

In [ ]:
print(f"Tokenized text: {tokenizer.convert_tokens_to_string(tokens)}")

In [ ]:
print(f"Vocab size is : {tokenizer.vocab_size}")

print(f"Model max length is : {tokenizer.model_max_length}")

print(f"Model input names are: {tokenizer.model_input_names}")

In [ ]:
ds

In [ ]:
dataset

In [ ]:
def tokenize_function(train_dataset):
    return tokenizer(train_dataset['Comment_cleaned'], padding='max_length', truncation=True)


tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset

train_dataset = tokenized_dataset['train']
eval_dataset = tokenized_dataset['valid']
test_dataset = tokenized_dataset['test']

In [ ]:
train_dataset

In [ ]:
test_dataset

In [ ]:
eval_dataset

In [ ]:
train_set = train_dataset.remove_columns([ "Comment_cleaned"]).with_format('tensorflow')

tf_eval_dataset = eval_dataset.remove_columns(["Comment_cleaned"]).with_format('tensorflow')

tf_test_dataset = test_dataset.remove_columns([ "Comment_cleaned"]).with_format('tensorflow')

In [ ]:
train_features = { x: train_set[x] for x in tokenizer.model_input_names  }

train_set_for_final_model = tf.data.Dataset.from_tensor_slices((train_features, train_set['Insult'] ))

train_set_for_final_model = train_set_for_final_model.shuffle(len(train_set)).batch(8)


eval_features = {x: tf_eval_dataset[x] for x in tokenizer.model_input_names}
val_set_for_final_model = tf.data.Dataset.from_tensor_slices((eval_features, tf_eval_dataset["Insult"]))
val_set_for_final_model = val_set_for_final_model.batch(8)

test_features = {x: tf_test_dataset[x] for x in tokenizer.model_input_names}
test_set_for_final_model = tf.data.Dataset.from_tensor_slices((test_features, tf_test_dataset["Insult"]))
test_set_for_final_model =test_set_for_final_model.batch(8)

In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained("GroNLP/hateBERT", num_labels=2)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=tf.metrics.SparseCategoricalAccuracy(),
)

In [ ]:
history = model.fit(train_set_for_final_model, validation_data=val_set_for_final_model, epochs=10)

In [ ]:
test_loss, test_acc = model.evaluate(test_set_for_final_model,verbose=2)
print('\nTest accuracy:', test_acc)

In [ ]:
from sklearn.metrics import classification_report

# Make predictions using the model on the test set
predictions = model.predict(test_set_for_final_model)
predicted_classes = np.argmax(predictions.logits, axis=1)

# Get the true classes from the test dataset
true_classes = tf_test_dataset["Insult"].numpy()

# Compute the classification report
class_labels = ["Neutral", "Insulting"]
classification_rep = classification_report(true_classes, predicted_classes, target_names=class_labels)

print("Classification Report:")
print(classification_rep)


In [ ]:
plt.plot(history.history['sparse_categorical_accuracy'])
plt.plot(history.history['val_sparse_categorical_accuracy'])
plt.title('model sparse categorical accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()


plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()


In [ ]:
predict_score_and_class_dict = {0: 'Neutral',
 1: 'Insulting'}
preds = model(tokenizer(["you can do it", "Stupid"],return_tensors="tf",padding=True,truncation=True))['logits']

print(preds)

class_preds = np.argmax(preds, axis=1)

for pred in class_preds:
  print(predict_score_and_class_dict[pred])

In [ ]:
from sklearn.metrics import confusion_matrix

# Predict classes for the test dataset
predictions = model.predict(test_set_for_final_model)
predicted_classes = np.argmax(predictions.logits, axis=1)

# Get the true classes from the test dataset
true_classes = tf_test_dataset["Insult"].numpy()

# Compute the confusion matrix
confusion_mat = confusion_matrix(true_classes, predicted_classes)

print("Confusion Matrix:")
print(confusion_mat)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Compute the confusion matrix
# confusion_mat = confusion_matrix(true_classes, predicted_classes)

# Define class labels
class_labels = ["Neutral", "Insulting"]

# Create a heatmap of the confusion matrix
plt.figure(figsize=(5, 5))
sns.heatmap(confusion_mat, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()
